In [0]:
# =============================================================================
# segmentation_diff  -  compare two per-case segmentation snapshots and explain every move.
# Reads snapshots produced by segmentation_snapshot (the test table OR exported parquet).
# Outputs: ADDED (in B not A), REMOVED (in A not B), MOVED (state changed),
#          VALID_FLIPPED (is_valid changed) - each with gate fields, so a changing total
#          is always explained case-by-case. READ ONLY. Writes nothing.
#
# SAFETY: reads only the TEST snapshot table / parquet files. Touches no dev/pipeline data.
# =============================================================================

In [0]:
# ---- CELL 0 : pick the two snapshots to compare ----
# Each source is either a snapshot_id from the test table, OR a parquet path.
SNAPSHOT_TABLE = "test_segmentation_snapshots"
A_SOURCE = "table:LABEL=postfix1290_20260819"   # forms: "table:ID=<snapshot_id>" | "table:LABEL=<data_cut_label>" | "parquet:<dbfs path>"
B_SOURCE = "table:LABEL=snap-20260814-073323-post2307b"          # the newer snapshot (e.g. after segmentation changes land)

from pyspark.sql import functions as F
from pyspark.sql.functions import *
REPORT=[]
def log(*a): REPORT.append(" ".join(str(x) for x in a))
def logdf(df,n=80):
    try: REPORT.append(df._jdf.showString(n,0,False))
    except Exception as e: REPORT.append(f"  (render fail: {str(e)[:120]})")

def load(src):
    kind,val = src.split(":",1)
    if kind=="parquet":
        return spark.read.parquet(val)
    df=spark.table(SNAPSHOT_TABLE)
    k,v = val.split("=",1)
    df=df.filter(col("snapshot_id")==v) if k=="ID" else df.filter(col("data_cut_label")==v)
    # if a label has multiple snapshots, keep the latest
    latest=df.agg(max("snapshot_datetime")).first()[0]
    return df.filter(col("snapshot_datetime")==latest)

A=load(A_SOURCE).select("appealReferenceNumber","assigned_state","is_valid","is_clone","casePrefix",
                        "caseStatus","outcome","statusDecisionDate","caseType","deptId","keyDate","decisionDate")
B=load(B_SOURCE).select("appealReferenceNumber","assigned_state","is_valid","is_clone","casePrefix",
                        "caseStatus","outcome","statusDecisionDate","caseType","deptId","keyDate","decisionDate")
nA,nB=A.count(),B.count()

In [0]:
# ---- CELL 1 : compute the diff ----
a=A.select("appealReferenceNumber").distinct(); b=B.select("appealReferenceNumber").distinct()
added   = B.join(a, "appealReferenceNumber", "left_anti")
removed = A.join(b, "appealReferenceNumber", "left_anti")
both = (A.select("appealReferenceNumber", col("assigned_state").alias("state_A"), col("is_valid").alias("valid_A"))
         .join(B.select("appealReferenceNumber", col("assigned_state").alias("state_B"), col("is_valid").alias("valid_B")),
               "appealReferenceNumber","inner"))
moved   = both.filter(~col("state_A").eqNullSafe(col("state_B")))
vflip   = both.filter(~col("valid_A").eqNullSafe(col("valid_B")))

log("="*74); log(f"SEGMENTATION DIFF   A={A_SOURCE}  ({nA})   ->   B={B_SOURCE}  ({nB})   net {nB-nA:+d}"); log("="*74)
log(f"ADDED (in B not A):     {added.count()}")
log(f"REMOVED (in A not B):   {removed.count()}")
log(f"MOVED state:            {moved.count()}")
log(f"VALID flipped:          {vflip.count()}")

In [0]:
# ---- CELL 2 : per-state net + the case lists ----
log("\n--- per-state net change ---")
sa=A.groupBy("assigned_state").count().withColumnRenamed("count","A")
sb=B.groupBy("assigned_state").count().withColumnRenamed("count","B")
per=(sa.join(sb,"assigned_state","full_outer").na.fill(0)
       .withColumn("delta", col("B")-col("A")).orderBy("assigned_state"))
logdf(per, 40)

log("\n--- REMOVED (in A, gone in B) with gate fields + is_clone ---")
logdf(removed.orderBy("assigned_state","appealReferenceNumber")
        .select("appealReferenceNumber","assigned_state","is_clone","is_valid","caseStatus","outcome","caseType","deptId","decisionDate"), 100)
log("\n--- ADDED (new in B) ---")
logdf(added.orderBy("assigned_state","appealReferenceNumber")
        .select("appealReferenceNumber","assigned_state","is_clone","is_valid","caseStatus","outcome","caseType","deptId","decisionDate"), 100)
log("\n--- MOVED state ---")
logdf(moved.orderBy("appealReferenceNumber"), 100)

In [0]:
# ---- CELL 3 : single print ----
log("\n"+"="*74)
log("READ: REMOVED with is_clone=true = benign clone drift/dedup; REMOVED with is_clone=false =")
log("      real cases that left segmentation -> check their gate fields (CaseType/DeptId 519,520/retention).")
full="\n".join(REPORT)
try:
    from datetime import datetime
    user=spark.sql("SELECT current_user()").first()[0]; ts=datetime.now().strftime("%Y%m%d_%H%M%S")
    folder=f"/Workspace/Users/{user}/Results/segmentation_diff/{ts}"; dbutils.fs.mkdirs(f"file:{folder}")
    p2=f"{folder}/segmentation_diff.txt"; open(p2,"w").write(full); full+=f"\n\n>>> saved to: {p2}"
except Exception as e: full+=f"\n(save failed: {str(e)[:80]})"
print(full)